# Unidad 16 · Casos de uso del análisis de sentimiento
## Caso de uso: Gestión de crisis

Este notebook forma parte de la entrega de la Unidad 16.  
El objetivo es diseñar un esquema de métricas para evaluar el impacto del análisis de sentimiento en un caso de uso empresarial.

### Objetivos
1. Reducir el tiempo de respuesta a menciones negativas.
2. Detectar tendencias que anticipen una crisis.
3. Mejorar la precisión en la clasificación de comentarios críticos.

## Carga de datos

El dataset `ejemplos.csv` contiene menciones simuladas de clientes con etiquetas de sentimiento (positivo/negativo) y marcas de tiempo de publicación/detección.  
Este archivo fue creado en la carpeta `/data` del repositorio y se cargó en Colab usando `pandas.read_csv`.



In [1]:
# Clonar tu repo desde GitHub
!git clone https://github.com/CODER-DataScience/Unidad16_CasosUsoSentimiento.git

# Entrar a la carpeta del repo
%cd Unidad16_CasosUsoSentimiento

# Verificar que el archivo está en data/
!ls data


Cloning into 'Unidad16_CasosUsoSentimiento'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 13 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 5.91 KiB | 1.18 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Unidad16_CasosUsoSentimiento
ejemplos.csv


In [5]:
import pandas as pd

# Cargar dataset simulado
df = pd.read_csv("data/ejemplos.csv")

# Verificar estructura del DataFrame
df.info()

# Mostrar primeras filas
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   texto                5 non-null      object
 1   sentimiento_real     5 non-null      object
 2   timestamp_publicado  5 non-null      int64 
 3   timestamp_detectado  5 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 292.0+ bytes


,texto,sentimiento_real,timestamp_publicado,timestamp_detectado
0,"El servicio está caído, pésimo soporte",negativo,1,2
1,"Muy buena atención, gracias!",positivo,2,3
2,"No funciona la app, estoy molesto",negativo,3,4
3,"Excelente experiencia, todo rápido",positivo,4,5
4,"Demasiados problemas, no lo recomiendo",negativo,5,7


## Métricas seleccionadas

1. **Recall clase negativa**  
   - Qué mide: proporción de menciones negativas correctamente detectadas.  
   - Fórmula: TP_neg / (TP_neg + FN_neg).  
   - Acción: asegura que no se escapen reclamos graves.  
   - Riesgo que reduce: falsos negativos que dejarían pasar comentarios críticos.

2. **Accuracy ponderada**  
   - Qué mide: porcentaje de clasificaciones correctas ajustado por desbalance de clases.  
   - Acción: evita métricas engañosas cuando hay más comentarios positivos que negativos.  
   - Riesgo que reduce: sobrevalorar la clase mayoritaria y subestimar la minoritaria.

3. **Tiempo de respuesta promedio**  
   - Qué mide: tiempo entre publicación y detección de menciones negativas.  
   - Acción: evalúa la eficiencia del pipeline de crisis.  
   - Riesgo que reduce: demoras en activar protocolos de respuesta.


## Cálculo de Recall clase negativa

El **recall** mide la capacidad del modelo para detectar correctamente los casos negativos.  
En este contexto, representa la proporción de menciones negativas que fueron clasificadas como negativas.  

- Fórmula: Recall = TP_neg / (TP_neg + FN_neg)  
- Riesgo que reduce: falsos negativos (comentarios críticos que pasan desapercibidos).  


In [6]:
from sklearn.metrics import recall_score

# Simulación de predicciones del modelo
df["sentimiento_pred"] = ["negativo", "positivo", "negativo", "positivo", "positivo"]

# Calcular Recall para la clase negativa
recall_neg = recall_score(df["sentimiento_real"], df["sentimiento_pred"], pos_label="negativo")
print("Recall clase negativa:", recall_neg)


Recall clase negativa: 0.6666666666666666


## Cálculo de Accuracy ponderada

La **accuracy** mide el porcentaje de clasificaciones correctas.  
En este caso, usamos una versión ponderada para evitar que el desbalance de clases (más positivos que negativos, o viceversa) distorsione la métrica.

- Fórmula: Accuracy = (TP + TN) / (Total)  
- Riesgo que reduce: sobrevalorar la clase mayoritaria y subestimar la minoritaria.


In [7]:
from sklearn.metrics import accuracy_score

# Calcular Accuracy
accuracy = accuracy_score(df["sentimiento_real"], df["sentimiento_pred"])
print("Accuracy:", accuracy)


Accuracy: 0.8


## Cálculo de Tiempo de respuesta promedio

El **tiempo de respuesta** mide la diferencia entre el momento en que se publica una mención negativa y el momento en que el sistema la detecta.  
Es una métrica de impacto operativo, directamente relacionada con la eficiencia del pipeline de crisis.

- Fórmula: Tiempo de respuesta = timestamp_detectado - timestamp_publicado  
- Riesgo que reduce: demoras en activar protocolos de respuesta rápida.


In [8]:
# Calcular tiempo de respuesta promedio
df["tiempo_respuesta"] = df["timestamp_detectado"] - df["timestamp_publicado"]
print("Tiempo de respuesta promedio:", df["tiempo_respuesta"].mean(), "minutos")


Tiempo de respuesta promedio: 1.2 minutos


## Interpretación de resultados

- El **recall negativo (0.66)** indica que el sistema detecta dos de cada tres menciones críticas.  
  Esto es aceptable, pero aún existe riesgo de falsos negativos que podrían dejar pasar reclamos graves.

- La **accuracy (0.80)** muestra un buen desempeño general del modelo, aunque debemos considerar que el dataset es pequeño y simulado.  
  En un escenario real, conviene usar accuracy ponderada para evitar sesgos por desbalance de clases.

- El **tiempo de respuesta promedio (1.2 minutos)** refleja un pipeline eficiente para la detección de crisis.  
  En la práctica, mantener tiempos bajos es clave para activar protocolos de respuesta rápida y minimizar impacto reputacional.

### Conclusión
El esquema de métricas permite evaluar tanto la calidad del modelo como el impacto operativo.  
En un contexto real, estas métricas deberían calcularse sobre datos de producción (tweets, tickets, reseñas) y revisarse periódicamente para garantizar que el sistema de gestión de crisis sea confiable y accionable.
